Dataset derivation

1. Uses Nemotron Personas for personalities
2. Uses 2021 M&P Survey as the baseline for the splits


Step 1: Import dataset from Nemotron

In [10]:
import pandas as pd
import numpy as np
import requests

url = "https://datasets-server.huggingface.co/rows?dataset=nvidia%2FNemotron-Personas-Singapore&config=default&split=train&offset=0&length=100"

response = requests.get(url)
response.raise_for_status()

rows = response.json()["rows"]
df = pd.DataFrame([item["row"] for item in rows])

df.head()

,uuid,professional_persona,sports_persona,arts_persona,travel_persona,culinary_persona,persona,cultural_background,skills_and_expertise,skills_and_expertise_list,...,hobbies_and_interests_list,career_goals_and_ambitions,sex,age,marital_status,education_level,occupation,industry,planning_area,country
0,d792702ec018494e8e49c69120759408,"Yi Peng Yong, known as Danelle, a 20‑year‑old ...","Yi Peng Yong, known as Danelle, enjoys playing...","Yi Peng Yong, known as Danelle, spends her eve...","Yi Peng Yong, known as Danelle, meticulously p...","Yi Peng Yong, known as Danelle, delights in co...","Yi Peng Yong, known as Danelle, blends a love ...",Yi Peng Yong grew up in the multicultural neig...,She has developed solid expertise in wholesale...,"['Inventory management', 'Supply chain coordin...",...,['Reading personal development and business bo...,She aims to deepen her knowledge of retail ope...,Female,20,Single,Polytechnic,Senior Official or Manager,Wholesale & Retail Trade,Sengkang,Singapore
1,2fe149f1c59e47daa5e6ae0881ffef93,"Charmaine, a 53‑year‑old experienced clerical ...","Charmaine, a devoted fan of Brunei DPMM FC, fo...","Charmaine, an avid admirer of Jack Neo’s comed...","Charmaine, who dreams of a serene Japan onsen ...","Charmaine, a passionate home cook, delights in...","Charmaine, a sociable 53‑year‑old office whiz ...",Charmaine grew up in a bilingual Chinese house...,Charmaine is an experienced clerical worker wi...,"['Data entry', 'Inventory management', 'Custom...",...,"['Karaoke', 'Cooking Cantonese cuisine', 'Gard...",Charmaine aims to advance to a senior administ...,Female,53,Married,Secondary,Clerical Worker,Wholesale & Retail Trade,Choa Chu Kang,Singapore
2,dd95514790e84ffa9973475cfc706660,"Betty, a 29‑year‑old policy analyst in the Min...",Betty is an avid follower of Red Bull Racing’s...,Betty’s artistic palate is shaped by her love ...,Betty meticulously plans her cherry‑blossom pi...,Betty excels at preparing classic Chinese dish...,Betty is a methodical policy pro who balances ...,Wei Qi Leong grew up in a middle‑class Chinese...,Betty has developed strong analytical and orga...,"['Policy analysis', 'Program management', 'Dat...",...,"['Reading non-fiction', 'Chinese calligraphy',...",Betty aims to advance to a senior policy advis...,Female,29,Single,University,Professional,Public Administration & Education Services,Clementi,Singapore
3,a2d66162e29f420aa93c670a41eb4fd2,"Jian Choy Ker, a retired hawker stall propriet...","Jian Choy Ker, though in his nineties, stays c...",Jian Choy Ker immerses himself in classic Cant...,Jian Choy Ker prefers relaxed staycations with...,"Jian Choy Ker, a seasoned hawker‑style cook, d...","Jian Choy Ker, a 91‑year‑old former hawker who...",Jian Choy Ker grew up in a traditional Chinese...,Through decades of work as a hawker stall prop...,"['Mandarin and Cantonese fluency', 'Basic Engl...",...,"['Mahjong', 'Chinese calligraphy', 'Gardening'...","Although retired, Jian aims to maintain good h...",Male,91,Married,Primary,Retired,NaN,Sengkang,Singapore
4,98a3e5752be446ebaca994b8a390b290,"Aixin Noy Yong, a dedicated homemaker turned c...","Aixin Noy Yong, a quiet admirer of the Golden ...","Aixin Noy Yong, an ardent fan of Andy Lau, spe...",Aixin Noy Yong dreams of a serene Japanese ons...,"Aixin Noy Yong, a master of traditional cookin...","Aixin Noy Yong, a 68‑year‑old tea‑tin collecto...",Aixin grew up in a traditional Chinese Singapo...,"Aixin is adept at traditional Chinese cooking,...","['Chinese culinary arts (dim sum, herbal soups...",...,"['Reading Buddhist texts', 'Listening to class...","Although she identifies as a homemaker, Aixin ...",Female,68,Single,Other Diploma,Homemaker,NaN,Kallang,Singapore


Step 2:
Extract our direct fields from dataset that agents need.

In [11]:
cols = [
    "age",
    "sex",
    "marital_status",
    "education_level",
    "occupation",
    "industry",
    "planning_area",
    "persona",
    "cultural_background",
    "career_goals_and_ambitions"
]

df = df[cols].copy()
df.head()

,age,sex,marital_status,education_level,occupation,industry,planning_area,persona,cultural_background,career_goals_and_ambitions
0,20,Female,Single,Polytechnic,Senior Official or Manager,Wholesale & Retail Trade,Sengkang,"Yi Peng Yong, known as Danelle, blends a love ...",Yi Peng Yong grew up in the multicultural neig...,She aims to deepen her knowledge of retail ope...
1,53,Female,Married,Secondary,Clerical Worker,Wholesale & Retail Trade,Choa Chu Kang,"Charmaine, a sociable 53‑year‑old office whiz ...",Charmaine grew up in a bilingual Chinese house...,Charmaine aims to advance to a senior administ...
2,29,Female,Single,University,Professional,Public Administration & Education Services,Clementi,Betty is a methodical policy pro who balances ...,Wei Qi Leong grew up in a middle‑class Chinese...,Betty aims to advance to a senior policy advis...
3,91,Male,Married,Primary,Retired,NaN,Sengkang,"Jian Choy Ker, a 91‑year‑old former hawker who...",Jian Choy Ker grew up in a traditional Chinese...,"Although retired, Jian aims to maintain good h..."
4,68,Female,Single,Other Diploma,Homemaker,NaN,Kallang,"Aixin Noy Yong, a 68‑year‑old tea‑tin collecto...",Aixin grew up in a traditional Chinese Singapo...,"Although she identifies as a homemaker, Aixin ..."


Step 3: Filter out age
Step 4: create age buckets for filtering

In [12]:
df = df[(df["age"] >= 21) & (df["age"] <= 45)].copy()
df.head()

def age_bucket(age):
    if 21 <= age <= 25:
        return "21-25"
    elif 26 <= age <= 30:
        return "26-30"
    elif 31 <= age <= 35:
        return "31-35"
    elif 36 <= age <= 40:
        return "36-40"
    elif 41 <= age <= 45:
        return "41-45"
    else:
        return np.nan

df["age_bucket"] = df["age"].apply(age_bucket)
df.head()

,age,sex,marital_status,education_level,occupation,industry,planning_area,persona,cultural_background,career_goals_and_ambitions,age_bucket
2,29,Female,Single,University,Professional,Public Administration & Education Services,Clementi,Betty is a methodical policy pro who balances ...,Wei Qi Leong grew up in a middle‑class Chinese...,Betty aims to advance to a senior policy advis...,26-30
5,39,Male,Married,University,Senior Official or Manager,Manufacturing,Sengkang,"Qijie Huat Chua, a night-owl puzzle enthusiast...",Qijie Huat Chua grew up in a Singaporean Chine...,He aims to advance to a C‑suite role such as C...,36-40
6,42,Female,Married,Post Secondary (Non-Tertiary),Homemaker,NaN,Jurong West,Wai Ting Soon is a habit‑driven planner who ca...,Wai Ting grew up in a traditional Chinese Sing...,While she values her role as a full‑time homem...,41-45
7,42,Male,Single,University,Professional,Financial & Insurance Services,Queenstown,Chongyong Ang balances a spreadsheet‑obsessed ...,Chongyong grew up in a typical Chinese Singapo...,Chongyong aims to progress to a senior managem...,41-45
9,31,Male,Single,University,Senior Official or Manager,Wholesale & Retail Trade,Woodlands,Yao Dar Teo (Daniel) is a data‑savvy retail st...,"Daniel grew up in Woodlands, a residential tow...",Daniel aims to advance to a senior director or...,31-35


Step 4: Standardise Education

In [13]:
def clean_education(x):
    x = str(x).lower()

    if any(k in x for k in ["secondary", "primary", "below", "ite", "n level", "o level"]):
        return "Secondary and below"

    elif any(k in x for k in ["diploma", "a level", "polytechnic", "junior college"]):
        return "Diploma / A-Level"

    elif any(k in x for k in ["degree", "bachelor", "master", "phd", "postgraduate", "university"]):
        return "Degree and above"

    else:
        return np.nan

df["education_bucket"] = df["education_level"].apply(clean_education)
df = df.dropna(subset=["education_bucket"])
df.head()


,age,sex,marital_status,education_level,occupation,industry,planning_area,persona,cultural_background,career_goals_and_ambitions,age_bucket,education_bucket
2,29,Female,Single,University,Professional,Public Administration & Education Services,Clementi,Betty is a methodical policy pro who balances ...,Wei Qi Leong grew up in a middle‑class Chinese...,Betty aims to advance to a senior policy advis...,26-30,Degree and above
5,39,Male,Married,University,Senior Official or Manager,Manufacturing,Sengkang,"Qijie Huat Chua, a night-owl puzzle enthusiast...",Qijie Huat Chua grew up in a Singaporean Chine...,He aims to advance to a C‑suite role such as C...,36-40,Degree and above
6,42,Female,Married,Post Secondary (Non-Tertiary),Homemaker,NaN,Jurong West,Wai Ting Soon is a habit‑driven planner who ca...,Wai Ting grew up in a traditional Chinese Sing...,While she values her role as a full‑time homem...,41-45,Secondary and below
7,42,Male,Single,University,Professional,Financial & Insurance Services,Queenstown,Chongyong Ang balances a spreadsheet‑obsessed ...,Chongyong grew up in a typical Chinese Singapo...,Chongyong aims to progress to a senior managem...,41-45,Degree and above
9,31,Male,Single,University,Senior Official or Manager,Wholesale & Retail Trade,Woodlands,Yao Dar Teo (Daniel) is a data‑savvy retail st...,"Daniel grew up in Woodlands, a residential tow...",Daniel aims to advance to a senior director or...,31-35,Degree and above


Step 5: Create Marital Group
Step 6: Split single agents into single vs dating

In [28]:
import json
import os
import time
import requests
from dotenv import load_dotenv

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("Set GEMINI_API_KEY or GOOGLE_API_KEY in .env")

GEMINI_MODEL = "gemini-2.0-flash-lite"
GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent"

RELATIONSHIP_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "relationship_status_classifier",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "relationship_status": {
                    "type": "string",
                    "enum": ["Single", "Dating", "Unclear"]
                },
                "confidence": {
                    "type": "number"
                },
                "evidence": {
                    "type": "string"
                }
            },
            "required": ["relationship_status", "confidence", "evidence"],
            "additionalProperties": False
        }
    }
}

def call_gemini(prompt, max_retries=5):
    for attempt in range(max_retries):
        response = requests.post(
            GEMINI_URL,
            params={"key": GEMINI_API_KEY},
            json={
                "contents": [
                    {
                        "role": "user",
                        "parts": [{"text": prompt}]
                    }
                ],
                "generationConfig": {
                    "temperature": 0,
                    "responseMimeType": "application/json"
                }
            },
            timeout=60
        )

        if response.status_code == 429:
            retry_after = response.headers.get("Retry-After")
            if retry_after and retry_after.isdigit():
                wait_seconds = int(retry_after)
            else:
                wait_seconds = min(30, 2 ** attempt)
            time.sleep(wait_seconds)
            continue

        response.raise_for_status()
        payload = response.json()
        text = payload["candidates"][0]["content"]["parts"][0]["text"].strip()
        result = json.loads(text)
        time.sleep(0.5)
        return result

    raise RuntimeError("Gemini rate limit persisted after retries")

def local_relationship_fallback(rows):
    fallback_results = []

    for row in rows:
        profile_text = " ".join([
            str(row.get("profile", {}).get("persona", "")),
            str(row.get("profile", {}).get("cultural_background", "")),
            str(row.get("profile", {}).get("career_goals_and_ambitions", ""))
        ]).lower()

        relationship_status = "Unclear"
        confidence = 0.2
        evidence = "No explicit relationship evidence found."

        if any(keyword in profile_text for keyword in ["boyfriend", "girlfriend", "partner", "dating", "seeing someone", "in a relationship"]):
            relationship_status = "Dating"
            confidence = 0.7
            evidence = "Relationship-related wording found in the profile text."
        elif any(keyword in profile_text for keyword in ["single", "not dating", "no partner"]):
            relationship_status = "Single"
            confidence = 0.6
            evidence = "Profile text suggests the agent is not currently dating."

        fallback_results.append({
            "index": row["index"],
            "relationship_status": relationship_status,
            "confidence": confidence,
            "evidence": evidence
        })

    return fallback_results

def classify_single_relationships(rows):
    records = []

    for row in rows:
        records.append({
            "index": int(row["index"]),
            "profile": {
                "age": row.get("age"),
                "gender": row.get("gender", row.get("sex")),
                "marital_status": row.get("marital_status"),
                "education_level": row.get("education_level"),
                "occupation": row.get("occupation"),
                "industry": row.get("industry"),
                "planning_area": row.get("planning_area"),
                "persona": row.get("general_persona", row.get("persona")),
                "cultural_background": row.get("cultural_background"),
                "career_goals_and_ambitions": row.get("career_goals_and_ambitions")
            }
        })

    prompt = f"""
You are classifying relationship status for a Singapore fertility-intention agent simulation.

Each agent is already known to be single or never-married in the structured data.
Your task is only to decide whether each profile gives evidence that the agent is currently dating.

Agent records:
{json.dumps(records, ensure_ascii=False, indent=2)}

Return one JSON array with one object per input record, in the same order.
Each object must include:
- "index": the original index from the input record
- "relationship_status": one of "Single", "Dating", or "Unclear"
- "confidence": a number between 0 and 1
- "evidence": a short explanation

Rules:
1. Do not invent a partner.
2. Do not infer dating from age, gender, education, occupation, industry, or planning area alone.
3. Only use persona text or structured fields as evidence.
4. If there is no relationship evidence, return "Unclear", not "Dating".
5. Return JSON only.
"""

    try:
        result = call_gemini(prompt)
        if isinstance(result, dict):
            result = [result]
        return result
    except RuntimeError:
        return local_relationship_fallback(records)

In [ ]:
single_df = df[df["marital_group"] == "Single"].copy()
single_records = single_df.reset_index().to_dict(orient="records")

relationship_results = pd.DataFrame(classify_single_relationships(single_records)).set_index("index")
relationship_results.head(100)

,relationship_status,confidence,evidence
index,,,
2,Unclear,0.2,No explicit relationship evidence found.
7,Unclear,0.2,No explicit relationship evidence found.
9,Unclear,0.2,No explicit relationship evidence found.
13,Unclear,0.2,No explicit relationship evidence found.
17,Unclear,0.2,No explicit relationship evidence found.


In [34]:
relationship_results.sort_values("relationship_status").tail()


,relationship_status,confidence,evidence
index,,,
13,Unclear,0.2,No explicit relationship evidence found.
9,Unclear,0.2,No explicit relationship evidence found.
7,Unclear,0.2,No explicit relationship evidence found.
35,Unclear,0.2,No explicit relationship evidence found.
94,Unclear,0.2,No explicit relationship evidence found.
